# 🚀 MLOps & Deployment Interview Handbook

ML-модель не приносит пользы, пока она "лежит в ноутбуке". Этот справочник посвящен тому, как обернуть ML-модель в API (FastAPI) и упаковать все это в контейнер (Docker), чтобы передать Production-команде.

---

## Блок 1: Виртуализация vs Контейнеризация

Частый теоретический вопрос на стыке Deployment и архитектуры:

### 🖥️ Виртуализация (Виртуальные машины - VM)
- Каждая виртуалка имеет **свою собственную полноценную Операционную Систему (Guest OS)** (например, Ubuntu, Windows).
- Очень "тяжелые" (весят гигабайты), долго запускаются (минуты).
- Полная изоляция безопасности (на уровне "железа" через Hypervisor).

### 🐳 Контейнеризация (Docker)
- Контейнеры **переиспользуют ядро Операционной Системы хоста (Host OS)**.
- Внутри контейнера лежат только библиотеки (зависимости) и код приложения.
- Очень "легкие" (миллисекунды на запуск, весят мегабайты).
- Работают везде одинаково (решает проблему "А на моем компе работало!").

---
## Блок 2: Docker и Docker-Compose

- **Docker:** Платформа для запуска контейнеров.
- **Dockerfile:** Текстовый файл-рецепт. Описывает, какой базовый образ взять (например, python:3.10), какие команды выполнить (`pip install`) и как запустить приложение.
- **Docker-Compose:** Инструмент для запуска сразу нескольких связанных контейнеров (например: Контейнер с FastAPI + Контейнер с Базой Данных PostgreSQL + Контейнер с Redis).

### 📝 Пример простейшего Dockerfile для ML:

```dockerfile
# 1. Берем легкий образ Python
FROM python:3.10-slim

# 2. Указываем рабочую директорию внутри контейнера
WORKDIR /app

# 3. Копируем зависимости и устанавливаем их
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 4. Копируем весь свой код (и веса модели)
COPY . .

# 5. Команда для запуска сервера
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### 🐙 Пример docker-compose.yml (Multi-container)
Используется для связки API с Redis (для кэша) или БД.

```yaml
version: '3.8'
services:
  api:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - redis
  redis:
    image: "redis:alpine"
```

### 🐙 Пример docker-compose.yml (Multi-container)
Используется для связки API с Redis (для кэша) или БД.

```yaml
version: '3.8'
services:
  api:
    build: .
    ports:
      - "8000:8000"
    depends_on:
      - redis
  redis:
    image: "redis:alpine"
```

---
## Блок 3: FastAPI (Лучший фреймворк для ML)

FastAPI стал стандартом де-факто для MLOps благодаря своей скорости (асинхронности) и автоматической валидации типов.

### 🔎 Основные концепции:
1. **Синхронные vs Асинхронные эндпоинты:**
   - `def my_endpoint():` (Синхронный) — выполнение заблокирует поток, пока код не отработает. ML-модели (Pytorch/Sklearn) почти всегда синхронные и не поддерживают `await` (процессорные вычисления).
   - `async def my_endpoint():` (Асинхронный) — не блокирует всю систему при обращении к базе данных или другим API (I/O вычисления).
2. **Pydantic:** Библиотека для валидации данных через типизацию Python. Ты описываешь классы (Модели Запроса и Ответа), а FastAPI сам следит, чтобы юзер прислал нужные данные (текст вместо числа вызовет ошибку 422).
3. **OpenAPI (Swagger):** Мгновенная автоматическая документация интерфейса, доступная прямо в браузере (по адресу `/docs`). Ничего не нужно писать руками!
4. **Health Check:** Обязательный эндпоинт (обычно `/health`), который просто отвечает `{"status": "ok"}`. Нужен, чтобы системы вроде Kubernetes понимали, не зависло ли приложение.

In [ ]:
# === Продвинутый FastAPI ML Boilerplate ===
from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel, validator
import time

app = FastAPI(title="Advanced ML API")

class PredictRequest(BaseModel):
    age: int
    income: float

    @validator('age')
    def age_must_be_positive(cls, v):
        if v <= 0:
            raise ValueError('Возраст должен быть положительным')
        return v

def log_request_to_db(data: dict):
    # Имитация долгой записи логов
    time.sleep(2)
    print(f"[Log] Запрос сохранен: {data}")

@app.post("/predict")
async def predict(request: PredictRequest, background_tasks: BackgroundTasks):
    # 1. Добавляем задачу в фон (не блокирует ответ пользователю)
    background_tasks.add_task(log_request_to_db, request.dict())
    
    # 2. Имитация предсказания
    result = "High Credit Score" if request.income > 50000 else "Low Credit Score"
    
    return {"prediction": result, "status": "logged_in_background"}

--- 
## Блок 4: Оркестрация и Мониторинг

### 🌬️ Apache Airflow (Оркестрация)
- **DAG (Directed Acyclic Graph):** Граф задач. Описывает последовательность этапов (Загрузка -> Обучение -> Валидация).
- **Operators:** Минимальные блоки (PythonOperator, BashOperator).
- **Scheduler:** Следит за временем запуска пайплайнов.

### 📉 Дрифт данных и модели (Monitoring)
- **Data Drift:** Изменение распределения входящих данных. 
- **Concept Drift:** Изменение связи между фичами и таргетом.
- **Metrics:** PSI (Population Stability Index) — помогает понять, насколько сильно разошлись два распределения.

--- 
## Блок 4: Оркестрация и Мониторинг

### 🌬️ Apache Airflow (Оркестрация)
- **DAG (Directed Acyclic Graph):** Граф задач. Описывает последовательность этапов (Загрузка -> Обучение -> Валидация).
- **Operators:** Минимальные блоки (PythonOperator, BashOperator).
- **Scheduler:** Следит за временем запуска пайплайнов.

### 📉 Дрифт данных и модели (Monitoring)
- **Data Drift:** Изменение распределения входящих данных. 
- **Concept Drift:** Изменение связи между фичами и таргетом.
- **Metrics:** PSI (Population Stability Index) — помогает понять, насколько сильно разошлись два распределения.

### 📝 Практические вопросы для самопроверки

1. Опишите своими словами, почему ML-модели чаще запускают внутри синхронных функций (`def`), а запросы в БД внутри асинхронных (`async def`)?
2. Что будет, если в эндпоинт `/predict` на место поля `income` пользователь отправит строку `"много денег"` вместо числа?
3. Как запустить контейнер Docker из готового `Dockerfile`? (ответ: `docker build -t my-app .` -> `docker run -p 8000:8000 my-app`).

---
## 🎯 Реальные вопросы с собеседований (ML Engineer / Python Developer)

### 🎤 Теоретические вопросы по архитектуре:
1. **Долгие запросы (Long-Polling/Task Queues):** Что произойдет, если инференс вашей ML модели занимает 10 секунд, и на FastAPI придет 1000 запросов одновременно? Как это исправить архитектурно? (Ожидается ответ про Celery / RabbitMQ / BackgroundTasks и возвращение `task_id`).
2. **Dockerfile CMD vs ENTRYPOINT:** В чем разница между `CMD ["python", "app.py"]` и `ENTRYPOINT ["python", "app.py"]`?
3. **States:** Что означает термин "Stateless сервис"? Почему микросервисы (и в частности ML-инференс) принято делать stateless, и где в таком случае хранить кэш сессий (ответ: Redis).
4. **Uvicorn Workers:** Зачем в команде запуска FastAPI передают флаг `--workers 4` (или работают через Gunicorn), если Python имеет GIL?

### ✅ Ответы на вопросы

<details>
<summary><b>Ответ 1: Model Registry (Registry ML-моделей)</b></summary>

Это хранилище версий. Когда инженеры обучают 50 моделей в день, им нужно понимать, какая модель сейчас на проде (Production), какая в стадии тестирования (Staging). Инструменты вроде **MLflow Registry** позволяют тегировать модели, хранить их гиперпараметры, артефакты и плавно управлять жизненным циклом (передать из staging в production).
</details>

<details>
<summary><b>Ответ 2: Data Drift vs Concept Drift</b></summary>

Модели в проде со временем "тупеют" из-за изменений в мире.
- **Data Drift:** Изменилось распределение входных данных (Внезапно больше пользователей из другого региона).
- **Concept Drift:** Изменилась сама связь между фичами и таргетом. То, что год назад считалось фродом, теперь стало нормой поведения.
**Решение:** Мониторинг распределений метрик и автоматическое дообучение (Retraining).
</details>

<details>
<summary><b>Ответ 3: CI/CD для ML (CT)</b></summary>

В отличие от классического ПО, в ML добавляется CT (Continuous Training).
- **CI (Continuous Integration):** Тестирование кода, линтеры, unit-тесты.
- **CD (Continuous Deployment):** Раскатка сервиса (Docker контейнера) на сервера с моделью.
- **CT (Continuous Training):** Пайплайн, который автоматически подхватывает новые данные из базы, обучает модель, проверяет метрики и, если они стали лучше, складывает модель в Registry для последующего деплоя.
</details>

---
## 🧠 Частые дополнительные вопросы (на понимание)

<details>
<summary><b>1. REST API vs gRPC для ML</b></summary>

Большинство ML сервисов (например `FastAPI`) общаются по **REST** (передают JSON-строки по HTTP). Это удобно и читаемо.
Но если нужна огромная скорость (много предсказаний миллисекундами), компании переходят на **gRPC**. Он передает данные в запакованном бинарном формате (Protobuf), что в несколько раз быстрее и требует меньше трафика.
</details>

<details>
<summary><b>2. Стратегии деплоя: Blue-Green vs Canary</b></summary>

Когда мы обновляем модель:
- **Blue-Green:** Мы поднимаем новый сервер (Green) с новой моделью. Если он работает исправно, мы моментально переключаем весь трафик со старого (Blue) на новый.
- **Canary (Канареечный):** Мы пускаем на новую модель только 5% трафика (пользователей). Следим за метриками. Если потерь нет, постепенно увеличиваем до 100%. Это минимизирует риски, если модель оказалась плохой.
</details>